# Customer Feedback Analysis using Natural Language Processing

**An end-to-end NLP pipeline built entirely on free and open-source models.**

**Dataset:** `Amazon_Reviews.csv` — ~21,000 public customer reviews of Amazon (2007–2024) with star rating,
title, text, date and country. Customers mostly write about delivery, refunds, customer service and accounts.

| Phase | What it does |
|---|---|
| 0 | Environment setup |
| 1 | Data collection |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | Text preprocessing |
| 4 | Baseline models (VADER, TF-IDF + ML) |
| 5 | Pretrained transformer (RoBERTa) |
| 6 | Fine-tuning DistilBERT |
| 7 | Model comparison & error analysis |
| 8 | Topic modeling (BERTopic) |
| 9 | Feedback categorization & aspect-based sentiment |
| 10 | Summarization & actionable insights |
| 11 | Export artifacts for the dashboard |

> **Run on Google Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.
> Total runtime on a T4 is roughly 40–60 minutes with the default settings.

---
## Phase 0 — Environment Setup
Install the libraries and set global configuration. All models used are free on the Hugging Face Hub.

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn nltk vaderSentiment \
    bertopic sentence-transformers wordcloud seaborn sentencepiece

In [ ]:
import os, re, random, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# ---------- Configuration ----------
SEED          = 42
DATA_PATH     = "Amazon_Reviews.csv"
TRAIN_CAP     = 5000     # max training reviews per class (large classes are down-sampled)
TRAIN_FLOOR   = 2500     # min training reviews per class (small classes are over-sampled)
EPOCHS        = 2        # DistilBERT fine-tuning epochs
MAX_LEN       = 256      # max tokens per review
OUTPUT_DIR    = "outputs"
MODEL_DIR     = "models/distilbert-feedback"
LABELS        = ["negative", "neutral", "positive"]
label2id      = {l: i for i, l in enumerate(LABELS)}
id2label      = {i: l for l, i in label2id.items()}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else -1
print("GPU available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "running on CPU")

---
## Phase 1 — Data Collection & Loading
**Upload `Amazon_Reviews.csv`** using the folder icon on the left in Colab (or run this cell and pick the file when asked).

Columns in the file: `Reviewer Name, Profile Link, Country, Review Count, Review Date, Rating, Review Title, Review Text, Date of Experience`.

What this phase does:
- Reads the file **robustly** — a few reviews in the raw file are broken across lines; those fragments are skipped.
- Extracts the star number from `"Rated 4 out of 5 stars"`.
- Builds the review text from **title + text** (the title is skipped when it is just a truncated copy of the text's first line).
- Maps stars to sentiment, removes duplicate reviews, and drops personal columns (name, profile link) we do not need.

| Stars | Sentiment |
|---|---|
| 1–2 | negative |
| 3 | neutral |
| 4–5 | positive |

In [ ]:
import csv

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Please upload Amazon_Reviews.csv")
        uploaded = files.upload()
        DATA_PATH = list(uploaded)[0]
    except ImportError:
        raise FileNotFoundError(f"Put {DATA_PATH} next to this notebook")

csv.field_size_limit(10**9)
with open(DATA_PATH, encoding="utf-8", newline="") as f:
    rows = list(csv.reader(f))
header, body = rows[0], rows[1:]
good = [r for r in body if len(r) == len(header)]
print(f"Complete reviews kept: {len(good)} | broken fragments skipped: {len(body) - len(good)}")

raw = pd.DataFrame(good, columns=header)
raw["stars"] = raw["Rating"].str.extract(r"Rated (\d)")[0].astype(int)

def build_text(title, text):
    title, text = str(title).strip(), str(text).strip()
    t = title.rstrip("…. ").lower()
    if not t or text.lower().startswith(t):
        return text
    return f"{title}. {text}"

raw["text"] = [build_text(a, b) for a, b in zip(raw["Review Title"], raw["Review Text"])]
raw["date"] = pd.to_datetime(raw["Review Date"], errors="coerce", utc=True)
raw["country"] = raw["Country"]
star_map = {1: "negative", 2: "negative", 3: "neutral", 4: "positive", 5: "positive"}
raw["sentiment"] = raw["stars"].map(star_map)

df = (raw[["text", "stars", "sentiment", "date", "country"]]
      .drop_duplicates(subset="text").reset_index(drop=True))
df.to_csv(f"{OUTPUT_DIR}/raw_feedback.csv", index=False)
print(df.shape)
print(df["sentiment"].value_counts())
df.head()

> **Note — class imbalance:** about 70% of the reviews are negative and only ~4% are neutral (3 stars).
> We handle this in Phase 3 by re-balancing the **training** set only, and we report **macro F1**,
> which gives each class equal weight. The test set keeps the real distribution.

---
## Phase 2 — Exploratory Data Analysis (EDA)
Understand the data before modeling: class balance, review length, and frequent words.

In [ ]:
df["n_words"] = df["text"].str.split().str.len()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="stars", hue="sentiment", hue_order=LABELS, dodge=False,
              palette=["#e74c3c", "#f1c40f", "#2ecc71"], ax=ax[0])
ax[0].set_title("Star rating distribution")
sns.boxplot(data=df, x="sentiment", y="n_words", order=LABELS, showfliers=False, ax=ax[1])
ax[1].set_title("Review length (words) by sentiment")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/eda_distribution.png", dpi=150); plt.show()

print(df.groupby("sentiment")["n_words"].describe().round(1))

In [ ]:
# Reviews over time, sentiment share per year, and top countries
df["year"] = df["date"].dt.year
fig, ax = plt.subplots(1, 3, figsize=(19, 4.5))
df["year"].value_counts().sort_index().plot(kind="bar", ax=ax[0], color="#3498db")
ax[0].set_title("Number of reviews per year")

recent = df[df["year"] >= 2015]
(pd.crosstab(recent["year"], recent["sentiment"], normalize="index")
   .reindex(columns=LABELS, fill_value=0)
   .plot(kind="bar", stacked=True, ax=ax[1], color=["#e74c3c", "#f1c40f", "#2ecc71"]))
ax[1].set_title("Sentiment share per year (2015+)"); ax[1].legend(loc="upper left", bbox_to_anchor=(1, 1))

df["country"].value_counts().head(10).plot(kind="barh", ax=ax[2], color="#9b59b6")
ax[2].invert_yaxis(); ax[2].set_title("Top 10 reviewer countries")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/eda_time_country.png", dpi=150); plt.show()

print(df.groupby("year")["stars"].mean().round(2).tail(10))

In [ ]:
from wordcloud import WordCloud, STOPWORDS

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, s, cmap in zip(axes, LABELS, ["Reds", "YlOrBr", "Greens"]):
    text = " ".join(df[df.sentiment == s]["text"].str.lower())
    wc = WordCloud(width=600, height=400, background_color="white",
                   stopwords=STOPWORDS | {"amazon"}, colormap=cmap, max_words=100).generate(text)
    ax.imshow(wc); ax.axis("off"); ax.set_title(f"{s.capitalize()} reviews", fontsize=14)
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/eda_wordclouds.png", dpi=150); plt.show()

---
## Phase 3 — Text Preprocessing
Steps: lowercasing → remove URLs/HTML/numbers/punctuation → tokenize → remove stopwords → lemmatize.

**Important:** negation words (*not, no, never*) are **kept**, otherwise "not good" becomes "good".

> Classical models (Phase 4) use the cleaned text. Transformers (Phases 5–6) use the **raw** text,
> because they understand punctuation, casing and context on their own.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
for pkg in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

NEGATIONS = {"not", "no", "nor", "never", "don't", "didn't", "isn't", "wasn't", "won't", "can't"}
STOP = (set(stopwords.words("english")) - NEGATIONS) | {"amazon"}   # "amazon" appears in almost every review
lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)        # URLs
    text = re.sub(r"<.*?>", " ", text)                      # HTML tags
    text = re.sub(r"n't", " not", text)                     # expand negation
    text = re.sub(r"[^a-z\s]", " ", text)                  # numbers, punctuation, emojis
    tokens = [lemmatizer.lemmatize(w) for w in text.split() if w not in STOP and len(w) > 1]
    return " ".join(tokens)

df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text"]].head()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["sentiment"], random_state=SEED)
train_df, val_df  = train_test_split(train_df, test_size=0.1, stratify=train_df["sentiment"], random_state=SEED)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]
print("Training set before re-balancing:", train_df["sentiment"].value_counts().to_dict())

def rebalance(d, cap=TRAIN_CAP, floor=TRAIN_FLOOR):
    # Down-sample big classes to `cap`, over-sample small classes up to `floor` (training set only)
    parts = []
    for _, g in d.groupby("sentiment"):
        if len(g) > cap:
            g = g.sample(cap, random_state=SEED)
        elif len(g) < floor:
            g = g.sample(floor, replace=True, random_state=SEED)
        parts.append(g)
    return pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

train_df = rebalance(train_df)
print("Training set after re-balancing: ", train_df["sentiment"].value_counts().to_dict())
print(f"Train: {len(train_df)} | Validation: {len(val_df)} | Test: {len(test_df)} (val/test keep the real distribution)")
df.to_csv(f"{OUTPUT_DIR}/clean_feedback.csv", index=False)

---
## Phase 4 — Baseline Models
1. **VADER** — rule-based lexicon, no training needed.
2. **TF-IDF + Logistic Regression / Multinomial Naive Bayes / Linear SVM** — classical machine learning.

These give us a reference point to judge whether transformers are worth the extra compute.

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

results = {}          # model name -> metrics
predictions = {}      # model name -> predicted labels on test set

def evaluate(name, y_true, y_pred, show=True):
    m = {"Accuracy": accuracy_score(y_true, y_pred),
         "Precision (macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
         "Recall (macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
         "F1 (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0)}
    results[name] = m
    predictions[name] = list(y_pred)
    if show:
        print(f"===== {name} =====")
        print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))
        cm = confusion_matrix(y_true, y_pred, labels=LABELS)
        ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(cmap="Blues")
        plt.title(name); plt.savefig(f"{OUTPUT_DIR}/cm_{name.replace(' ', '_').replace('+','')}.png", dpi=120)
        plt.show()
    return m

In [ ]:
# ----- 4.1 VADER -----
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()

def vader_label(text, pos=0.05, neg=-0.05):
    c = vader.polarity_scores(text)["compound"]
    return "positive" if c >= pos else "negative" if c <= neg else "neutral"

evaluate("VADER", test_df["sentiment"], test_df["text"].apply(vader_label))

In [ ]:
# ----- 4.2 TF-IDF + classical classifiers -----
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train = tfidf.fit_transform(train_df["clean_text"])
X_test  = tfidf.transform(test_df["clean_text"])

classical = {
    "TF-IDF + Naive Bayes": MultinomialNB(),
    "TF-IDF + Logistic Regression": LogisticRegression(max_iter=2000, C=2.0),
    "TF-IDF + Linear SVM": LinearSVC(C=0.5),
}
for name, model in classical.items():
    model.fit(X_train, train_df["sentiment"])
    evaluate(name, test_df["sentiment"], model.predict(X_test))

In [ ]:
# Most influential words learned by Logistic Regression (great for the report)
lr = classical["TF-IDF + Logistic Regression"]
features = np.array(tfidf.get_feature_names_out())
for i, cls in enumerate(lr.classes_):
    top = features[np.argsort(lr.coef_[i])[-15:]][::-1]
    print(f"{cls:>9}: {', '.join(top)}")

---
## Phase 5 — Pretrained Transformer (No Training)
Model: [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest)
— a RoBERTa model already trained for 3-class sentiment. We use it *zero-shot* on our reviews.

In [ ]:
from transformers import pipeline

roberta = pipeline("sentiment-analysis",
                   model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                   device=DEVICE, truncation=True, max_length=512)

roberta_out = roberta(test_df["text"].tolist(), batch_size=32)
roberta_pred = [o["label"].lower() for o in roberta_out]
evaluate("RoBERTa (pretrained)", test_df["sentiment"], roberta_pred)

---
## Phase 6 — Fine-tuning DistilBERT on Our Data
Model: [`distilbert-base-uncased`](https://huggingface.co/distilbert-base-uncased) — a smaller, faster BERT (66M parameters).
We fine-tune it on our labeled reviews so it learns the domain. Takes ~15–25 min on a free T4 GPU.

In [ ]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

BASE_MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def to_hf(d):
    ds = Dataset.from_dict({"text": d["text"].tolist(),
                            "label": d["sentiment"].map(label2id).tolist()})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True)

hf_train, hf_val, hf_test = to_hf(train_df), to_hf(val_df), to_hf(test_df)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro")}

import math
steps_per_epoch = math.ceil(len(hf_train) / 16)          # 16 = train batch size
WARMUP_STEPS = int(0.1 * steps_per_epoch * EPOCHS)       # warm up over the first 10% of training

args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=WARMUP_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=hf_train, eval_dataset=hf_val,
                  data_collator=DataCollatorWithPadding(tokenizer),
                  compute_metrics=compute_metrics)
trainer.train()

In [ ]:
# Training curve
logs = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(figsize=(8, 4))
logs.dropna(subset=["loss"]).plot(x="step", y="loss", ax=ax, label="train loss")
if "eval_loss" in logs:
    logs.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, label="validation loss", marker="o")
ax.set_title("DistilBERT fine-tuning loss"); plt.savefig(f"{OUTPUT_DIR}/training_curve.png", dpi=120); plt.show()

# Evaluate on the held-out test set
test_logits = trainer.predict(hf_test).predictions
distil_pred = [id2label[i] for i in np.argmax(test_logits, axis=-1)]
evaluate("DistilBERT (fine-tuned)", test_df["sentiment"], distil_pred)

# Save for the dashboard
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print("Saved fine-tuned model to", MODEL_DIR)

---
## Phase 7 — Model Comparison & Error Analysis
Compare every model on the same test set, then study where the best model fails.

In [ ]:
comparison = pd.DataFrame(results).T.sort_values("F1 (macro)", ascending=False).round(4)
comparison.to_csv(f"{OUTPUT_DIR}/model_comparison.csv")
display(comparison)

comparison["F1 (macro)"].plot(kind="barh", figsize=(9, 4), color="#3498db")
plt.xlabel("Macro F1 score"); plt.title("Model comparison (test set)"); plt.gca().invert_yaxis()
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/model_comparison.png", dpi=150); plt.show()

In [ ]:
best_model = comparison.index[0]
print("Best model:", best_model)

err = test_df[["text", "sentiment"]].copy()
err["predicted"] = predictions[best_model]
wrong = err[err["sentiment"] != err["predicted"]]
print(f"Misclassified: {len(wrong)} / {len(err)} ({len(wrong)/len(err):.1%})\n")
print(pd.crosstab(wrong["sentiment"], wrong["predicted"]))

pd.set_option("display.max_colwidth", 300)
wrong.sample(min(10, len(wrong)), random_state=SEED)

**What to write in your report:** most errors usually involve the *neutral* class — it is only ~4% of this dataset,
and 3-star reviews mix praise and complaints —
sarcasm ("Great, another 2-hour wait"), and mixed opinions ("Food was great, service was terrible").
This motivates Phase 9 — aspect-based sentiment.

---
## Phase 8 — Topic Modeling (What Are Customers Talking About?)
**BERTopic** groups reviews into topics using sentence embeddings
([`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)) + UMAP + clustering.

Two choices made for this dataset:
- **630 reviews have the placeholder text "Review text not found"** (only a title was written). The placeholder is
  removed and very short reviews are left out, otherwise BERTopic groups reviews by that phrase instead of by meaning.
- **K-Means clustering with a fixed number of topics** instead of BERTopic's default HDBSCAN. Almost every review
  is about Amazon complaints, so HDBSCAN puts nearly all reviews into one giant topic. K-Means forces the
  reviews to split into `N_TOPICS` distinct themes.

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.cluster import KMeans

TOPIC_SAMPLE = 6000
N_TOPICS     = 12

# Remove the "Review text not found" placeholder and skip very short reviews
topic_src = df.copy()
topic_src["text"] = (topic_src["text"]
                     .str.replace(r"\.?\s*Review\s?text not found\.?", "", case=False, regex=True)
                     .str.strip())
topic_src = topic_src[topic_src["text"].str.split().str.len() >= 8]
topic_df = topic_src.sample(min(TOPIC_SAMPLE, len(topic_src)), random_state=SEED).reset_index(drop=True)
print("Reviews used for topic modeling:", len(topic_df))

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(topic_df["text"].tolist(), show_progress_bar=True, batch_size=64)

STOP_TOPIC = list(ENGLISH_STOP_WORDS | {"amazon", "dont", "im", "ive", "just", "like", "get", "got"})
topic_model = BERTopic(
    hdbscan_model=KMeans(n_clusters=N_TOPICS, random_state=SEED, n_init=10),
    vectorizer_model=CountVectorizer(stop_words=STOP_TOPIC, ngram_range=(1, 2), min_df=1),
    verbose=True)   # min_df=1: BERTopic treats each topic as one document
topics, _ = topic_model.fit_transform(topic_df["text"].tolist(), embeddings)
topic_df["topic"] = topics

info = topic_model.get_topic_info()
info.to_csv(f"{OUTPUT_DIR}/topics.csv", index=False)
print("Number of topics:", (info["Topic"] != -1).sum())
info.head(15)

In [ ]:
topic_model.visualize_barchart(top_n_topics=8, n_words=6)

In [ ]:
# Which topics drive negative feedback?
topic_sent = (topic_df[topic_df.topic != -1]
              .groupby("topic")["sentiment"].value_counts(normalize=True)
              .unstack().reindex(columns=LABELS, fill_value=0).fillna(0))
topic_names = info.set_index("Topic")["Name"]
topic_sent.index = [topic_names.get(t, t)[:40] for t in topic_sent.index]
topic_sent = topic_sent.sort_values("negative", ascending=False).head(12)

topic_sent.plot(kind="barh", stacked=True, color=["#e74c3c", "#f1c40f", "#2ecc71"], figsize=(10, 6))
plt.title("Sentiment share per topic (most negative at top)"); plt.gca().invert_yaxis()
plt.xlabel("Share of reviews"); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/topic_sentiment.png", dpi=150); plt.show()

---
## Phase 9 — Feedback Categorization & Aspect-Based Sentiment Analysis (ABSA)
1. **Zero-shot categorization** with [`facebook/bart-large-mnli`](https://huggingface.co/facebook/bart-large-mnli):
   tag each review with business categories *without any training data*.
2. **ABSA** with [`yangheng/deberta-v3-base-absa-v1.1`](https://huggingface.co/yangheng/deberta-v3-base-absa-v1.1):
   find the sentiment toward each specific aspect inside one review.

The categories below are chosen for Amazon customer feedback. Edit `CATEGORIES` to add or remove any.

In [ ]:
from transformers import pipeline

CATEGORIES = ["delivery", "customer service", "refund and returns", "account and payment",
              "prime membership", "product quality", "price"]

zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=DEVICE)

CAT_SAMPLE = 400   # zero-shot is slow; categorize a sample
cat_df = test_df.sample(min(CAT_SAMPLE, len(test_df)), random_state=SEED).reset_index(drop=True)
zs_out = zero_shot([t[:1000] for t in cat_df["text"]], candidate_labels=CATEGORIES,
                   multi_label=True, batch_size=16)

cat_df["categories"] = [[l for l, s in zip(o["labels"], o["scores"]) if s > 0.6] or [o["labels"][0]]
                        for o in zs_out]
cat_df[["text", "sentiment", "categories"]].head(8)

In [ ]:
cat_long = cat_df.explode("categories").reset_index(drop=True)
pd.crosstab(cat_long["categories"], cat_long["sentiment"]).reindex(columns=LABELS, fill_value=0).plot(
    kind="bar", color=["#e74c3c", "#f1c40f", "#2ecc71"], figsize=(10, 4))
plt.title("Sentiment by feedback category"); plt.ylabel("Number of reviews"); plt.xticks(rotation=30)
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/category_sentiment.png", dpi=150); plt.show()

In [ ]:
absa = pipeline("text-classification", model="yangheng/deberta-v3-base-absa-v1.1", device=DEVICE)

def aspect_sentiment(review, aspects):
    out = {}
    for a in aspects:
        r = absa({"text": review[:1500], "text_pair": a})
        r = r[0] if isinstance(r, list) else r
        out[a] = (r["label"].lower(), round(r["score"], 3))
    return out

examples = [
    "The product quality is great but delivery was a week late and customer service kept me on hold.",
    "Prime shipping is fast and the refund was quick, though prices have gone up a lot.",
]
for ex in examples:
    print(ex)
    for a, (lab, sc) in aspect_sentiment(ex, ["delivery", "customer service", "refund", "price", "product quality"]).items():
        print(f"   {a:<13} -> {lab:<8} ({sc})")
    print()

---
## Phase 10 — Summarization & Actionable Insights
Use [`facebook/bart-large-cnn`](https://huggingface.co/facebook/bart-large-cnn) to summarize the negative reviews in each
of the most-complained-about topics, producing a short "what's going wrong" report for management.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

sum_tok = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
sum_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn").to(
    "cuda" if torch.cuda.is_available() else "cpu")

def summarize(texts, max_words=180):
    joined = " ".join(" ".join(t.split()[:60]) for t in texts)          # trim each review
    inputs = sum_tok(joined, return_tensors="pt", truncation=True, max_length=1024).to(sum_model.device)
    ids = sum_model.generate(**inputs, max_length=max_words, min_length=40, num_beams=4,
                             no_repeat_ngram_size=3, early_stopping=True)
    return sum_tok.decode(ids[0], skip_special_tokens=True)

insights = []
neg_topics = (topic_df[(topic_df.topic != -1) & (topic_df.sentiment == "negative")]
              ["topic"].value_counts().head(5).index)
for t in neg_topics:
    neg = topic_df[(topic_df.topic == t) & (topic_df.sentiment == "negative")]
    neg = neg[neg["text"].str.split().str.len() >= 20]          # detailed reviews summarize better
    reviews = neg["text"].head(12).tolist()
    s = summarize(reviews)
    insights.append({"topic": topic_names.get(t, t), "n_negative": len(reviews), "summary": s})
    print(f"\n### Topic: {topic_names.get(t, t)}\n{s}")

pd.DataFrame(insights).to_csv(f"{OUTPUT_DIR}/insights.csv", index=False)

**Rerunning only Stage 4 in a new session?** Run this cell to download the new Stage 4 files,
then copy them into the `outputs` folder of your project (replace the old ones).

In [ ]:
import shutil, os
os.makedirs("stage4_outputs", exist_ok=True)
for f in ["topics.csv", "topic_sentiment.png", "category_sentiment.png", "insights.csv"]:
    if os.path.exists(f"{OUTPUT_DIR}/{f}"):
        shutil.copy(f"{OUTPUT_DIR}/{f}", "stage4_outputs")
shutil.make_archive("stage4_outputs", "zip", "stage4_outputs")
try:
    from google.colab import files
    files.download("stage4_outputs.zip")
except ImportError:
    print("Created stage4_outputs.zip")

---
## Phase 11 — Export Artifacts for the Dashboard
Package the fine-tuned model and outputs so `app.py` (the Streamlit dashboard) can use them.
Download `project_outputs.zip` and place its contents next to `app.py`.

In [ ]:
# Works even after a Colab restart: everything is read back from the saved files
import os, json, shutil
import pandas as pd

OUTPUT_DIR = "outputs"
MODEL_DIR  = "models/distilbert-feedback"
assert os.path.isdir(MODEL_DIR), "Model folder missing - the session was reset. Rerun the notebook from the top."

comparison = pd.read_csv(f"{OUTPUT_DIR}/model_comparison.csv", index_col=0)
summary = {
    "dataset": "Amazon_Reviews.csv",
    "best_model": comparison["F1 (macro)"].idxmax(),
    "comparison": comparison.to_dict(),
    "categories": ["delivery", "customer service", "refund and returns", "account and payment",
                   "prime membership", "product quality", "price"],
}
with open(f"{OUTPUT_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Best model:", summary["best_model"])

os.makedirs("export", exist_ok=True)
shutil.copytree("models", "export/models", dirs_exist_ok=True)
shutil.copytree(OUTPUT_DIR, "export/outputs", dirs_exist_ok=True)
shutil.make_archive("project_outputs", "zip", "export")
print("Created project_outputs.zip:", round(os.path.getsize("project_outputs.zip") / 1e6), "MB")

try:
    from google.colab import files
    files.download("project_outputs.zip")
except ImportError:
    pass

**Optional - keep a copy in Google Drive.** Colab deletes all files when the session ends;
Drive keeps them. Run the cell below and allow access when asked.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy("project_outputs.zip", "/content/drive/MyDrive/project_outputs.zip")
    print("Saved to Google Drive -> MyDrive/project_outputs.zip")
except ImportError:
    print("Not running in Colab - skip this cell.")

---
## Conclusion
- Classical TF-IDF models are fast and surprisingly strong baselines.
- Fine-tuned transformers give the best accuracy, especially on nuanced and neutral feedback.
- Topic modeling + aspect-based sentiment turn raw feedback into **specific, actionable** problems
  (for this dataset: delivery failures, refunds and customer-service handling dominate the complaints).
- Class imbalance (very few neutral reviews) is the main limitation; collecting more 3-star reviews would help most.

**Future scope:** Hindi/Hinglish feedback with `ai4bharat/IndicBERTv2`, real-time streaming analysis,
sarcasm detection, and a Q&A chatbot over feedback using a free open LLM (e.g., Llama or Mistral via Ollama).